# Ночное обучение детектора мусора

В VS Code выберите **Select Kernel → Python Environments → `.venv\python.exe`**. Затем нажмите **Run All**. Для ночного запуска отключите переход компьютера в сон.

Выберите режим в первой ячейке.

| Режим | Что запускается |
|---|---|
| `train` | Продолжает прерванные серии, обучает новые и сравнивает лучшие веса для суши и подводных кадров. Готовые серии используются повторно. |
| `evaluate` | Сравнивает текущую модель приложения и завершённые серии на validation. |
| `smoke` | Заново обучает каждую пробную модель одну эпоху на суше при 960 px и batch 8. Веса и отчёт сохраняются в отдельных папках для каждого запуска. |

При пересоздании `.venv` установите поддержку Jupyter командой `.\.venv\python.exe -m pip install -r requirements-notebook.txt`.

## 1. Настройки

In [ ]:
from pathlib import Path

MODE = "train"  # "train", "evaluate" или "smoke"
PROJECT_ROOT = None  # Если ноутбук открыт вне проекта, укажите полный путь к проекту.
PROFILES = ("land", "underwater")
INCLUDE_YOLO26 = True
YOLO26_EPOCHS = 80
YOLO26_PATIENCE = 15
YOLO26_BATCH = 8
WORKERS = 1

## 2. Модели и контрольные точки

В очереди по пять вариантов на профиль. Текущая модель оценивается в размере из конфигурации приложения, остальные — при 960 px. Для новых запусков используется seed 42. В режиме `smoke` проверяются YOLO11s и обе YOLO26 на суше.

Для YOLO11 действует прежний рецепт обучения — до 80 эпох и ранняя остановка после 15 эпох без улучшения validation mAP50–95. При возобновлении YOLO11 восстанавливаются веса, оптимизатор и номер эпохи, а счётчик ранней остановки начинается заново. Для YOLO26 лимиты задаются в первой ячейке. Чтобы продолжить прерванную серию, запустите `train` с теми же настройками.

In [ ]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
from dataclasses import dataclass
from datetime import datetime
from math import isfinite
from time import perf_counter
from urllib.parse import quote

import yaml
from filelock import FileLock, Timeout
from IPython.display import Markdown, display
from litter_detector.experiments import validate_existing_run

PROFILE_LABELS = {"land": "Суша", "underwater": "Под водой"}
if MODE not in {"train", "evaluate", "smoke"}:
    raise ValueError("Выберите MODE: train, evaluate или smoke.")
if not PROFILES or not set(PROFILES) <= PROFILE_LABELS.keys() or len(set(PROFILES)) != len(PROFILES):
    raise ValueError("Выберите один или оба профиля: land, underwater.")
if min(YOLO26_EPOCHS, YOLO26_PATIENCE, YOLO26_BATCH) < 1 or WORKERS < 0:
    raise ValueError("Эпохи, patience и batch должны быть положительными, WORKERS — от 0.")

start = Path(PROJECT_ROOT or Path.cwd()).resolve()
ROOT = next((path for path in (start, *start.parents)
             if (path / "scripts/experiment.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Укажите PROJECT_ROOT в первой ячейке.")

smoke_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f") if MODE == "smoke" else None
OUTPUT = ROOT / "outputs/overnight_training"
if smoke_id:
    OUTPUT = ROOT / "outputs/overnight_training_smoke" / smoke_id
PRETRAINED = ROOT / "models/pretrained"
PYTHON = {
    "yolo11": ROOT / ".venv/python.exe",
    "yolo26": ROOT / ".cache/conda-envs/yolo26/python.exe",
}
active_profiles = ("land",) if MODE == "smoke" else PROFILES


@dataclass(frozen=True)
class Candidate:
    profile: str
    name: str
    arch: str
    imgsz: int = 960
    batch: int = 8
    epochs: int = 80
    patience: int = 15
    reference: Path | None = None

    @property
    def family(self):
        return "yolo26" if self.arch.startswith("yolo26") else "yolo11"

    @property
    def folder(self):
        return ROOT / "models/experiments" / self.profile / self.name

    @property
    def weights(self):
        return self.reference or self.folder / "weights/best.pt"

    @property
    def recipe(self):
        return dict(profile=self.profile, name=self.name, arch=self.arch, imgsz=self.imgsz,
                    batch=self.batch, epochs=self.epochs, patience=self.patience, seed=42, workers=WORKERS)


candidates = []
yolo26_models = ("yolo26n", "yolo26s") if INCLUDE_YOLO26 else ()
for profile in active_profiles:
    if MODE == "smoke":
        candidates.extend(Candidate(profile, f"notebook_smoke_{arch}_{smoke_id}", arch, epochs=1, patience=1)
                          for arch in ("yolo11s", *yolo26_models))
        continue
    config = yaml.safe_load((ROOT / "configs" / f"{profile}.yaml").read_text(encoding="utf-8"))
    candidates.extend([
        Candidate(profile, "current", "Текущая", int(config["imgsz"]), reference=ROOT / config["model"]),
        Candidate(profile, "n960_long", "yolo11n", batch=16 if profile == "underwater" else 8),
        Candidate(profile, "s_compare", "yolo11s"),
    ])
    for arch in yolo26_models:
        name = f"overnight_{arch}_960_e{YOLO26_EPOCHS}_p{YOLO26_PATIENCE}_b{YOLO26_BATCH}"
        candidates.append(Candidate(profile, name, arch, batch=YOLO26_BATCH,
                                    epochs=YOLO26_EPOCHS, patience=YOLO26_PATIENCE))

## 3. Проверка данных и окружений

До обучения проверяются файлы validation и доступность CUDA в нужных окружениях. Валидация внутри каждого профиля использует один список `data/v2/<профиль>/val.txt`. Выборка `test` остаётся для отдельной итоговой проверки.

In [ ]:
def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def child_environment(family):
    config_dir = OUTPUT / "ultralytics" / family
    font = Path(os.environ.get("WINDIR", r"C:\Windows")) / "Fonts/arial.ttf"
    local_font = config_dir / "Ultralytics/Arial.ttf"
    local_font.parent.mkdir(parents=True, exist_ok=True)
    if font.is_file() and not local_font.is_file():
        shutil.copyfile(font, local_font)
    # VS Code can set the inline backend, which the separate YOLO26 environment cannot import.
    return dict(os.environ, PYTHONIOENCODING="utf-8", YOLO_OFFLINE="true", YOLO_AUTOINSTALL="false",
                YOLO_CONFIG_DIR=str(config_dir), MPLCONFIGDIR=str(config_dir / "matplotlib"),
                MPLBACKEND="Agg")


validation_inputs = {}
for profile in active_profiles:
    dataset = ROOT / "data/v2" / profile
    validation_list = dataset / "val.txt"
    paths = [Path(line) for line in validation_list.read_text(encoding="utf-8").splitlines() if line.strip()]
    if not paths:
        raise ValueError(f"Пустой список validation: {validation_list}")
    for path in paths:
        label = dataset / "labels/val" / f"{path.stem}.txt"
        if not path.is_file() or not label.is_file():
            raise FileNotFoundError(f"Проверьте пару файлов: {path} и {label}")
    validation_inputs[profile] = (sha256(validation_list), sha256(dataset / "manifest.json"), len(paths))
    print(f"{PROFILE_LABELS[profile]} · {len(paths)} кадров validation")


def preflight():
    families = set()
    for candidate in candidates:
        if MODE == "evaluate" and candidate.reference is None and not (candidate.folder / "summary.json").is_file():
            continue
        families.add(candidate.family)
    print(f"Ядро: {sys.executable}")
    probe = (
        "import torch, ultralytics, matplotlib, litter_detector; "
        "assert matplotlib.get_backend().lower() == 'agg', matplotlib.get_backend(); "
        "assert torch.cuda.is_available(), 'CUDA недоступна'; "
        "print(f'Ultralytics {ultralytics.__version__} · {torch.cuda.get_device_name(0)}')"
    )
    for family in sorted(families):
        python = PYTHON[family]
        if not python.is_file():
            raise FileNotFoundError(f"Не найдено окружение {family}: {python}")
        result = subprocess.run([str(python), "-c", probe], cwd=PRETRAINED, env=child_environment(family),
                                capture_output=True, encoding="utf-8", errors="replace", timeout=60)
        if result.returncode:
            raise RuntimeError(f"Ошибка окружения {family} ({python})\n{result.stdout}{result.stderr}")
        print(f"{family.upper()} · {result.stdout.strip().splitlines()[-1]}")

## 4. Запуск очереди

Модели обучаются и проверяются по одной. Во время обучения ячейка выводит сохранённые эпохи. `comparison.csv` обновляется после каждой серии. Подробности доступны в `logs/`, а метрики и предсказания — в `validation/` внутри папки отчёта.

In [ ]:
FIELDS = ["profile", "model", "series", "imgsz", "epochs", "epochs_limit", "map50_95", "delta_pp",
          "map50", "f1", "precision", "recall", "small_recall", "confidence", "median_ms", "p95_ms",
          "status", "source", "weights", "log", "error"]


def epochs_completed(folder):
    path = folder / "results.csv"
    if not path.is_file():
        return None
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return max((int(row["epoch"]) for row in csv.DictReader(stream, skipinitialspace=True)
                    if row.get("epoch", "").strip().isdigit()), default=None)


def run_script(candidate, stage, script, args, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = [str(PYTHON[candidate.family]), "-u", str(ROOT / "scripts" / script), *map(str, args)]
    print(f"  {stage} · {log_path.relative_to(ROOT)}", flush=True)
    started = perf_counter()
    with log_path.open("a", encoding="utf-8") as log:
        log.write(f"\n[{datetime.now().astimezone().isoformat(timespec='seconds')}]\n")
        log.flush()
        with subprocess.Popen(command, cwd=PRETRAINED, env=child_environment(candidate.family),
                              stdout=log, stderr=subprocess.STDOUT) as process:
            try:
                while True:
                    try:
                        code = process.wait(timeout=30)
                        break
                    except subprocess.TimeoutExpired:
                        epoch = None
                        if stage == "train":
                            try:
                                epoch = epochs_completed(candidate.folder)
                            except OSError:
                                pass  # The trainer may be writing the CSV at this instant.
                        progress = f" · эпохи {epoch}/{candidate.epochs}" if epoch is not None else ""
                        print(f"  {stage} · {round(perf_counter() - started)} с{progress}", flush=True)
            except KeyboardInterrupt:
                process.terminate()
                raise
    if code:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        detail = next((line.strip() for line in reversed(lines) if line.strip()), "")
        raise RuntimeError(f"Код {code} · {detail[:400]}")


def evaluate(candidate, log_path):
    destination = OUTPUT / "validation" / candidate.profile / f"{candidate.name}.json"
    expected_hash = sha256(candidate.weights)
    run_script(candidate, "val", "predict_v2.py", [candidate.weights, candidate.profile, "--split", "val",
               "--imgsz", candidate.imgsz, "--output", destination], log_path)
    report = json.loads(destination.read_text(encoding="utf-8"))
    inputs = (report["file_list_sha256"], report["manifest_sha256"], len(report["records"]))
    if (inputs != validation_inputs[candidate.profile]
            or report["weight_sha256"] != expected_hash or sha256(candidate.weights) != expected_hash
            or (report["profile"], report["split"], report["imgsz"], report["sliced"])
            != (candidate.profile, "val", candidate.imgsz, False)):
        raise ValueError("Отчёт оценки не совпадает с заданными данными, весами или параметрами.")

    point = max(report["thresholds"], key=lambda row: row["f1"])
    metrics = {key: point[key] for key in ("f1", "precision", "recall", "small_recall", "confidence")}
    metrics.update(map50_95=report["ap"]["map50_95"], map50=report["ap"]["ap50"],
                   median_ms=report["median_ms"], p95_ms=report["p95_ms"])
    if any(not isfinite(value) for value in metrics.values() if value is not None):
        raise ValueError("Отчёт оценки содержит некорректную метрику.")
    return dict(metrics, weights=candidate.weights.relative_to(ROOT).as_posix())


def run_candidate(candidate, row):
    folder = candidate.folder
    log_folder = OUTPUT / "logs" / candidate.profile
    log_path = log_folder / f"{candidate.name}_train.log"
    stage = "обучения"
    try:
        if candidate.reference is None:
            row["epochs"] = epochs_completed(folder)
            summary = validate_existing_run(folder, candidate.recipe, ROOT)
            if summary is None:
                if MODE == "evaluate":
                    row["status"] = "Ожидает обучения"
                    return
                checkpoint = folder / "weights/last.pt"
                if not checkpoint.is_file() and not (PRETRAINED / f"{candidate.arch}.pt").is_file():
                    raise FileNotFoundError(f"Не найдены стартовые веса {candidate.arch}.pt")
                row["source"] = "Возобновлена" if checkpoint.is_file() else "Обучена"
                args = [candidate.profile, candidate.name]
                for key in ("arch", "imgsz", "batch", "epochs", "patience", "seed", "workers"):
                    args.extend((f"--{key}", candidate.recipe[key]))
                run_script(candidate, "train", "experiment.py", args, log_path)
                row["epochs"] = epochs_completed(folder)
                if not (folder / "summary.json").is_file():
                    raise FileNotFoundError(folder / "summary.json")
            else:
                row["source"] = "Готовые веса"
        else:
            row["source"] = "Из приложения"
            row["epochs"] = epochs_completed(candidate.weights.parent.parent)

        stage = "оценки"
        log_path = log_folder / f"{candidate.name}_val.log"
        row.update(evaluate(candidate, log_path), status="Оценена")
        print(f"  mAP50–95 = {row['map50_95']:.4f}", flush=True)
    except (OSError, ValueError, KeyError, RuntimeError) as error:
        row.update(status=f"Ошибка {stage}", error=str(error))
        print(f"  {row['status']} · {error}", flush=True)
    finally:
        if log_path.is_file():
            row["log"] = log_path.relative_to(ROOT).as_posix()


def rank_results(rows):
    ranked = []
    for profile in active_profiles:
        group = [row.copy() for row in rows if row["profile"] == profile]
        scored = sorted((row for row in group if row["status"] == "Оценена"),
                        key=lambda row: (-row["map50_95"], row["median_ms"], row["series"]))
        pending = [row for row in group if row["status"] != "Оценена"]
        baseline = next((row["map50_95"] for row in scored if row["series"] == "current"), None)
        if baseline is not None:
            for row in scored:
                row["delta_pp"] = 100 * (row["map50_95"] - baseline)
        if scored:
            if MODE == "smoke":
                scored[0]["status"] = "Лидер пробного запуска"
            else:
                scored[0]["status"] = "Лидер среди оценённых" if pending else "Рекомендована"
        ranked.extend(scored + pending)
    return ranked


def save_comparison(rows):
    destination = OUTPUT / "comparison.csv"
    temporary = destination.with_suffix(".csv.tmp")
    with temporary.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rank_results(rows))
    temporary.replace(destination)


results = [dict(profile=c.profile, model=c.arch.replace("yolo", "YOLO"), series=c.name, imgsz=c.imgsz,
                epochs_limit=None if c.reference else c.epochs, status="В очереди") for c in candidates]
lock_path = ROOT / "outputs/overnight_training/queue.lock"
lock_path.parent.mkdir(parents=True, exist_ok=True)
try:
    with FileLock(str(lock_path), timeout=0):
        OUTPUT.mkdir(parents=True, exist_ok=True)
        preflight()
        for index, (candidate, row) in enumerate(zip(candidates, results), start=1):
            print(f"[{index}/{len(candidates)}] {PROFILE_LABELS[candidate.profile]} · {row['model']}", flush=True)
            try:
                run_candidate(candidate, row)
            except KeyboardInterrupt:
                row["status"] = "Прервана"
                save_comparison(results)
                raise
            save_comparison(results)
except Timeout as error:
    raise RuntimeError("Очередь уже запущена в другом ядре этого проекта.") from error
print(f"Таблица сохранена в {OUTPUT.relative_to(ROOT) / 'comparison.csv'}")

## 5. Сравнительная таблица

Для каждого профиля побеждает наибольший **mAP50–95** на validation. При точном равенстве выигрывает модель с меньшей медианной задержкой. **Δ к текущей** — разница mAP50–95 с моделью приложения в процентных пунктах. Если часть серий ещё не оценена, таблица показывает предварительного лидера. Строка с ошибкой ведёт к логу, если он доступен.

`scripts/predict_v2.py` считает AP по предсказаниям с confidence от 0.05. F1 и Recall мелких объектов показаны при пороге, максимизирующем F1 на validation. Мелкие объекты имеют площадь меньше 32 × 32 пикселей на подготовленном кадре. В CSV также доступны Precision, Recall, mAP50 и 95-й перцентиль задержки. Результаты пробного запуска относятся к одной эпохе обучения.

In [ ]:
columns = [
    ("profile", "Профиль"), ("model", "Модель"), ("imgsz", "Вход, px"), ("epochs", "Эпохи"),
    ("map50_95", "mAP50–95"), ("delta_pp", "Δ к текущей, п.п."), ("f1", "F1"),
    ("small_recall", "Recall мелких"), ("confidence", "Порог"), ("median_ms", "мс/кадр"),
    ("status", "Решение"), ("weights", "Веса"),
]


def table_value(row, key):
    value = row.get(key)
    if value is None:
        return "—"
    if key == "profile":
        return PROFILE_LABELS[value]
    if key == "epochs" and row["epochs_limit"] is not None:
        return f"{value}/{row['epochs_limit']}"
    if key == "weights":
        return f"[best.pt]({quote(value, safe='/')})"
    if key == "status":
        if value.startswith(("Лидер", "Рекомендована")):
            return f"**{value}**"
        if row.get("error") and row.get("log"):
            return f"[{value}]({quote(row['log'], safe='/')})"
    if key == "delta_pp":
        return f"{value:+.2f}" if value else "0.00"
    if isinstance(value, float):
        digits = {"map50_95": 4, "median_ms": 1, "confidence": 2}.get(key, 3)
        return f"{value:.{digits}f}"
    return str(value)


table = ["| " + " | ".join(label for _, label in columns) + " |",
         "| " + " | ".join("---" for _ in columns) + " |"]
table.extend("| " + " | ".join(table_value(row, key) for key, _ in columns) + " |"
             for row in rank_results(results))
display(Markdown("\n".join(table)))